Python port of biodiversity_code.R

Kenya insect-diversity SDM: compares 4 environmental-predictor scenarios
(GeoMAD, GeoMAD_Climate_Canopy_HF, TCI, TCI_Climate_Canopy_HF) using a
Random Forest presence/background model, with a full set of
publication-quality diagnostic figures per scenario plus a cross-scenario
comparison at the end.

The input paths
(/outputs/Shapefiles/Kenya.shp, the 4 scenario raster folders).
The translation is faithful to the R logic, but
a few pieces don't have an exact library-for-library equivalent; see the
inline notes at each:
  - randomForest's importance="MeanDecreaseAccuracy" -> sklearn's
    permutation_importance (built-in .feature_importances_ is Gini-based,
    i.e. closer to MeanDecreaseGini, not MeanDecreaseAccuracy)
  - usdm::vifstep -> the same iterative-VIF-elimination function used
    elsewhere in this project
  - ggplot2/patchwork/corrplot styling -> matplotlib, approximated, not
    pixel-identical
  - terra::cellSize (geodesic area) -> simple planar pixel-area from the
    transform, correct if the scenario rasters are in a projected CRS in
    metres (the "_250M" folder names suggest they are) but not
    geodetically exact if they turn out to be in a geographic CRS

In [ ]:
import os
import pickle
import platform
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.transform
from rasterio.warp import Resampling, reproject, calculate_default_transform
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, auc as sk_auc
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

## 1. SETUP

In [ ]:
PALETTE_MAIN = ["#1B9E77", "#D95F02", "#7570B3", "#E7298A", "#66A61E", "#E6AB02"]
PALETTE_SCENARIO = ["#E66B4B", "#4B8DE6", "#E6D14B", "#8B4BE6"]

OCCURRENCE_FILE = "/outputs/Shapefiles/Kenya_cleaned_minus_hf_presence_points.csv"
AOI_FILE = "/outputs/Shapefiles/Kenya.shp"
VERSION_NUMBER = "5"
OUTPUT_BASE_DIR = f"/outputs/Scenario_Comparison_Results/Version_{VERSION_NUMBER}"

SCENARIO_PATHS = [
    "/outputs/Kenya/Kenya_Geomad_2020_2023_250M",
    "/outputs/Kenya/Kenya_Geomad_SPECTRAL_2020_2023_250M",
    "/outputs/Kenya/Kenya_TCI_2020_2023_250M",
    "/outputs/Kenya/Kenya_TCI_SPECTRAL_2020_2023_250M/RESAMPLED",
]
SCENARIO_NAMES = ["GeoMAD", "GeoMAD_Climate_Canopy_HF", "TCI", "TCI_Climate_Canopy_HF"]

# The "GeoMAD" scenario (index 0) loads via authenticated datacube.load()
# instead of a local raster folder - see DATACUBE_SCENARIOS / the branch in
# section 4a. 
DATACUBE_SCENARIOS = {"GeoMAD"}
GEOMAD_DATACUBE_YEARS = [2020, 2021, 2022, 2023]  # averaged - approximates the "2020_2023" multi-year composite
GEOMAD_DATACUBE_BANDS = [
    "blue", "green", "red", "red_edge_1", "red_edge_2", "red_edge_3",
    "nir", "nir_narrow", "swir_1", "swir_2", "emad", "smad", "bcmad",
]
GEOMAD_DATACUBE_RES_M = 250  # matches the "_250M" in the other scenarios' folder names
GEOMAD_DATACUBE_CRS = "EPSG:6933"  

RNG_SEED = 42
np.random.seed(RNG_SEED)
TRAIN_TEST_SPLIT = 0.8
VIF_THRESHOLD = 2000  # matches the R script's th=2000 - very permissive, VIF rarely excludes anything at this level

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)


def log(*args):
    print(*args)


def apply_publication_style(ax, title=None, subtitle=None):
    """Approximates theme_publication(): bold axis titles, black border,
    light gridlines, subtitle as a smaller grey line under the title."""
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(0.8)
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")
    ax.grid(True, color="grey", alpha=0.25, linewidth=0.25)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, fontweight="bold", loc="left", fontsize=12)
    if subtitle:
        ax.text(0.0, 1.02, subtitle, transform=ax.transAxes, color="grey", fontsize=9)


performance_summary = []  # list of dicts, one per scenario -> DataFrame at the end

## 2. HELPER FUNCTIONS

In [ ]:
def fetch_geomad_datacube_kenya(bbox_4326):
    """Loads GeoMAD directly via datacube.load() for the 'GeoMAD' scenario,
    instead of reading local raster files. Averages gm_s2_annual across
    GEOMAD_DATACUBE_YEARS to approximate a multi-year composite, since
    dc.load() with a time range returns one slice per year rather than a
    pre-blended composite. Returns the same {name: (array, transform, crs,
    nodata)} shape the file-reading branch produces, so every downstream
    step (alignment, masking, VIF, etc.) works unchanged."""
    import datacube

    minx, miny, maxx, maxy = bbox_4326
    dc = datacube.Datacube(app="biodiversity_code")

    yearly = {band: [] for band in GEOMAD_DATACUBE_BANDS}
    transform = crs = None
    for year in GEOMAD_DATACUBE_YEARS:
        log(f"  Loading gm_s2_annual {year} from the DE Africa datacube...")
        ds = dc.load(
            product="gm_s2_annual", time=str(year), x=(minx, maxx), y=(miny, maxy),
            output_crs=GEOMAD_DATACUBE_CRS, resolution=(-GEOMAD_DATACUBE_RES_M, GEOMAD_DATACUBE_RES_M),
            resampling="bilinear", measurements=GEOMAD_DATACUBE_BANDS,
        )
        if transform is None:
            transform, crs = ds.geobox.affine, GEOMAD_DATACUBE_CRS
        for band in GEOMAD_DATACUBE_BANDS:
            yearly[band].append(ds[band].squeeze().values.astype("float32"))

    log(f"  Averaging {len(GEOMAD_DATACUBE_YEARS)} years ({GEOMAD_DATACUBE_YEARS}) per band...")
    band_arrays = {}
    for band in GEOMAD_DATACUBE_BANDS:
        stacked = np.stack(yearly[band])
        with np.errstate(invalid="ignore"):
            mean_arr = np.nanmean(stacked, axis=0)
        band_arrays[band] = (mean_arr, transform, crs, None)

    return band_arrays


def get_spec_sens_threshold(observed, predicted):
    """Youden's J statistic: threshold maximizing (sensitivity + specificity - 1)."""
    fpr, tpr, thresholds = roc_curve(observed, predicted)
    j = tpr - fpr
    return thresholds[np.argmax(j)]


def get_confusion_metrics(actual, predicted_binary):
    actual = np.asarray(actual, dtype=int)
    predicted_binary = np.asarray(predicted_binary, dtype=int)

    tp = int(np.sum((predicted_binary == 1) & (actual == 1)))
    tn = int(np.sum((predicted_binary == 0) & (actual == 0)))
    fp = int(np.sum((predicted_binary == 1) & (actual == 0)))
    fn = int(np.sum((predicted_binary == 0) & (actual == 1)))

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_score = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

    return {
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Sensitivity": sensitivity, "Specificity": specificity, "Accuracy": accuracy,
        "Precision": precision, "F1_Score": f1_score,
    }


def vifstep(df, threshold=5.0):
    """Iterative VIF elimination, mirroring usdm::vifstep: drop the variable
    with the highest VIF above threshold, repeat until all VIFs are below it."""
    vars_left = list(df.columns)
    excluded = []
    while len(vars_left) > 1:
        X = sm.add_constant(df[vars_left])
        vifs = pd.Series(
            [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])], index=vars_left,
        )
        if vifs.max() <= threshold:
            break
        worst = vifs.idxmax()
        vars_left.remove(worst)
        excluded.append(worst)
    return vars_left, excluded


def partial_dependence(model, train_df, predictor_cols, variable, grid_points=50):
    """Partial dependence for one variable: hold all others at the value of
    the first training row, sweep `variable` across its observed range."""
    var_range = (train_df[variable].min(), train_df[variable].max())
    grid_values = np.linspace(var_range[0], var_range[1], grid_points)

    base_row = train_df[predictor_cols].iloc[[0]]
    pred_data = pd.concat([base_row] * grid_points, ignore_index=True)
    pred_data[variable] = grid_values

    pred_probs = model.predict_proba(pred_data[predictor_cols])[:, 1]
    return pd.DataFrame({"Variable": variable, "Value": grid_values, "Probability": pred_probs})

## 3. LOAD OCCURRENCE AND AOI DATA

In [ ]:
log("Loading occurrence data...")
occ_data = pd.read_csv(OCCURRENCE_FILE)
if "longitude" not in occ_data.columns or "latitude" not in occ_data.columns:
    raise ValueError("Occurrence data must contain 'longitude' and 'latitude' columns")

occ_sf_base = gpd.GeoDataFrame(
    occ_data, geometry=gpd.points_from_xy(occ_data.longitude, occ_data.latitude), crs="EPSG:4326"
)
log("Total occurrence points loaded:", len(occ_sf_base))

log("Loading AOI shapefile...")
aoi_base = gpd.read_file(AOI_FILE)
if aoi_base.crs is None:
    aoi_base = aoi_base.set_crs("EPSG:4326")

occ_sf_base = occ_sf_base.to_crs(aoi_base.crs)

points_in_aoi = occ_sf_base.geometry.intersects(aoi_base.geometry.union_all())
if points_in_aoi.sum() < len(occ_sf_base):
    log(f"Filtering {len(occ_sf_base) - points_in_aoi.sum()} points outside AOI...")
    occ_sf_base = occ_sf_base[points_in_aoi.to_numpy()]

n_presence = len(occ_sf_base)
log(f"Cleaned presence points in AOI: {n_presence}\n")

# Needed by the datacube-backed GeoMAD scenario (dc.load() takes lon/lat)
aoi_bounds_4326 = aoi_base.to_crs("EPSG:4326").total_bounds

## 4. SCENARIO LOOP

In [ ]:
for i, (scenario_name, rasters_path) in enumerate(zip(SCENARIO_NAMES, SCENARIO_PATHS), start=1):

    log("\n" + "=" * 70)
    log(f"SCENARIO ({i}/{len(SCENARIO_NAMES)}): {scenario_name}")
    log("=" * 70)

    scenario_output_dir = os.path.join(OUTPUT_BASE_DIR, scenario_name)
    os.makedirs(scenario_output_dir, exist_ok=True)

    # --- 4a. Load and align rasters ---
    if scenario_name in DATACUBE_SCENARIOS:
        log(f"Loading {scenario_name} via datacube.load() (Sandbox) instead of {rasters_path}...")
        try:
            band_arrays = fetch_geomad_datacube_kenya(aoi_bounds_4326)
        except Exception as e:
            log(f"ERROR: datacube fetch failed for {scenario_name}: {e}. Skipping scenario...")
            continue
    else:
        log("Loading environmental rasters...")
        raster_files = [
            os.path.join(rasters_path, f) for f in os.listdir(rasters_path)
            if f.lower().endswith((".tif", ".asc", ".img"))
        ] if os.path.isdir(rasters_path) else []

        if not raster_files:
            log(f"WARNING: No raster files found in {rasters_path}. Skipping scenario...")
            continue

        band_arrays = {}   # name -> (array, transform, crs, nodata)
        for f in raster_files:
            try:
                with rasterio.open(f) as src:
                    name = src.descriptions[0] or os.path.splitext(os.path.basename(f))[0]
                    if not name:
                        name = os.path.splitext(os.path.basename(f))[0]
                    arr = src.read(1).astype("float32")

                    # population_mean -> clamp to 1000, rename to population_impact
                    f_name = os.path.splitext(os.path.basename(f))[0]
                    if name == "population_mean" or f_name == "population_mean":
                        log("  [Processing] Clamping 'population_mean' (>1000 -> 1000) and renaming to 'population_impact'...")
                        arr = np.clip(arr, None, 1000)
                        name = "population_impact"

                    band_arrays[name] = (arr, src.transform, src.crs, src.nodata)
            except Exception as e:
                log(f"  Could not read {f}: {e}")

    if not band_arrays:
        log(f"ERROR: Could not load any rasters for {scenario_name}")
        continue

    initial_vars = list(band_arrays.keys())
    var_display = ", ".join(initial_vars[:3]) + (" ..." if len(initial_vars) > 3 else "")
    log(f"Loaded {len(initial_vars)} variables: {var_display}")

    # Reference grid: prefer a band named like B02/B03/B04 (Sentinel-2 style), else the first
    ref_name = next((n for n in initial_vars if any(b in n for b in ("B02", "B03", "B04"))), initial_vars[0])
    ref_arr, ref_transform, ref_crs, ref_nodata = band_arrays[ref_name]
    ref_height, ref_width = ref_arr.shape
    log(f"Reference CRS: {ref_crs}")

    def align_to_ref(arr, transform, crs):
        if crs == ref_crs and arr.shape == ref_arr.shape and transform == ref_transform:
            return arr
        dest = np.full((ref_height, ref_width), np.nan, dtype="float32")
        reproject(
            source=arr, destination=dest,
            src_transform=transform, src_crs=crs,
            dst_transform=ref_transform, dst_crs=ref_crs,
            resampling=Resampling.bilinear,
        )
        return dest

    aligned = {}
    for name, (arr, transform, crs, nodata) in band_arrays.items():
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)
        if name == ref_name:
            aligned[name] = arr
        else:
            log(f"  Aligning {name}...")
            aligned[name] = align_to_ref(arr, transform, crs)

    env_names_all = list(aligned.keys())
    env_stack = np.stack([aligned[n] for n in env_names_all])

    # --- 4b. Crop/mask to AOI, then VIF variable selection ---
    log("Cropping, masking, and variable selection...")

    aoi_reproj = aoi_base.to_crs(ref_crs)
    stack_profile = {
        "driver": "GTiff", "height": ref_height, "width": ref_width, "count": len(env_names_all),
        "dtype": "float32", "crs": ref_crs, "transform": ref_transform, "nodata": np.nan,
    }
    from rasterio.io import MemoryFile
    with MemoryFile() as memfile:
        with memfile.open(**stack_profile) as tmp:
            tmp.write(env_stack.astype("float32"))
        with memfile.open() as tmp:
            masked, masked_transform = rio_mask(tmp, aoi_reproj.geometry, crop=True)

    # Audit empty layers
    valid_layers = [not np.all(np.isnan(masked[k])) for k in range(masked.shape[0])]
    empty_layers = [n for n, v in zip(env_names_all, valid_layers) if not v]
    if empty_layers:
        log("[Audit] Empty layers removed:", ", ".join(empty_layers))
    keep_idx = [k for k, v in enumerate(valid_layers) if v]
    env_stack_masked = masked[keep_idx]
    env_names_masked = [env_names_all[k] for k in keep_idx]

    # Extract values at occurrence locations
    occ_sf_reproj = occ_sf_base.to_crs(ref_crs)
    rows, cols = rasterio.transform.rowcol(masked_transform, occ_sf_reproj.geometry.x, occ_sf_reproj.geometry.y)
    rows, cols = np.array(rows), np.array(cols)
    h, w = env_stack_masked.shape[1], env_stack_masked.shape[2]
    in_bounds = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)

    occ_env_values = pd.DataFrame(np.nan, index=range(len(occ_sf_reproj)), columns=env_names_masked)
    for j, name in enumerate(env_names_masked):
        vals = np.full(len(occ_sf_reproj), np.nan)
        vals[in_bounds] = env_stack_masked[j][rows[in_bounds], cols[in_bounds]]
        occ_env_values[name] = vals

    na_counts = occ_env_values.isna().sum()
    vars_with_na = na_counts[na_counts > 0]
    if len(vars_with_na) > 0:
        log("[Audit] Variables with NAs at occurrence points:")
        for v, c in vars_with_na.items():
            log(f"  {v}: {c}/{len(occ_env_values)} points")

    occ_env_complete = occ_env_values.dropna()
    log(f"Complete cases for VIF: {len(occ_env_complete)}/{len(occ_env_values)} points")

    var_check = occ_env_complete.var()
    zero_var_cols = var_check[(var_check == 0) | var_check.isna()].index.tolist()
    if zero_var_cols:
        log("[Audit] Zero-variance variables removed:", ", ".join(zero_var_cols))
        occ_env_complete = occ_env_complete.drop(columns=zero_var_cols)

    if occ_env_complete.shape[1] > 1:
        retained_names, excluded_vars = vifstep(occ_env_complete, threshold=VIF_THRESHOLD)
    else:
        retained_names, excluded_vars = list(occ_env_complete.columns), []

    log("VIF Step completed. Excluded:", ", ".join(excluded_vars) if excluded_vars else "None")

    sel_idx = [env_names_masked.index(v) for v in retained_names]
    env_stack_selected = env_stack_masked[sel_idx]
    selected_predictor_names = retained_names

    accounted_for = set(selected_predictor_names) | set(excluded_vars) | set(empty_layers) | set(zero_var_cols)
    unaccounted = [v for v in initial_vars if v not in accounted_for]

    log("------ VARIABLE ACCOUNTING ------")
    log(f"Retained ({len(selected_predictor_names)}): {', '.join(selected_predictor_names)}")
    log(f"VIF Excluded ({len(excluded_vars)}): {', '.join(excluded_vars) if excluded_vars else 'None'}")
    log(f"Empty/NA Layers ({len(empty_layers)}): {', '.join(empty_layers) if empty_layers else 'None'}")
    log("---------------------------------")

    # --- 4c. Background sampling and train/test split ---
    log(f"Sampling {n_presence} background points...")

    valid_mask = np.all(np.isfinite(env_stack_selected), axis=0)
    valid_rows, valid_cols = np.where(valid_mask)
    rng = np.random.default_rng(RNG_SEED)
    n_bg = n_presence - 250
    bg_choice = rng.choice(len(valid_rows), size=min(n_bg, len(valid_rows)), replace=False)
    bg_r, bg_c = valid_rows[bg_choice], valid_cols[bg_choice]

    def extract_at_rc(stack, rr, cc, names):
        df = pd.DataFrame(index=range(len(rr)), columns=names, dtype="float64")
        for j, name in enumerate(names):
            df[name] = stack[j][rr, cc]
        return df

    occ_vals = extract_at_rc(env_stack_selected, rows, cols, selected_predictor_names)
    occ_vals = occ_vals[in_bounds].reset_index(drop=True)
    occ_vals["presence"] = 1

    bg_vals = extract_at_rc(env_stack_selected, bg_r, bg_c, selected_predictor_names)
    bg_vals["presence"] = 0

    combined_data = pd.concat([occ_vals, bg_vals], ignore_index=True).dropna()
    combined_data["presence"] = combined_data["presence"].astype(int)

    idx_0 = combined_data.index[combined_data.presence == 0].to_numpy()
    idx_1 = combined_data.index[combined_data.presence == 1].to_numpy()
    rng.shuffle(idx_0)
    rng.shuffle(idx_1)
    train_idx = np.concatenate([
        idx_0[: int(np.floor(TRAIN_TEST_SPLIT * len(idx_0)))],
        idx_1[: int(np.floor(TRAIN_TEST_SPLIT * len(idx_1)))],
    ])
    test_idx = combined_data.index.difference(train_idx)

    train_data = combined_data.loc[train_idx].reset_index(drop=True)
    test_data = combined_data.loc[test_idx].reset_index(drop=True)
    pred_cols = [c for c in train_data.columns if c != "presence"]

    log(f"Training data: {len(train_data)} (presence: {(train_data.presence==1).sum()}, absence: {(train_data.presence==0).sum()})")
    log(f"Test data: {len(test_data)} (presence: {(test_data.presence==1).sum()}, absence: {(test_data.presence==0).sum()})")

    # --- 4d. Train Random Forest ---
    log("Training Random Forest model...")
    rf_model = RandomForestClassifier(
        n_estimators=500, max_features=max(1, int(np.floor(np.sqrt(len(pred_cols))))),
        random_state=RNG_SEED,
    )
    rf_model.fit(train_data[pred_cols], train_data["presence"])

    with open(os.path.join(scenario_output_dir, "rf_model.pkl"), "wb") as f:
        pickle.dump(rf_model, f)
    with open(os.path.join(scenario_output_dir, "selected_predictors.pkl"), "wb") as f:
        pickle.dump(selected_predictor_names, f)

    # --- 4e. Model evaluation ---
    p_test_num = test_data["presence"].to_numpy()
    rf_pred_prob = rf_model.predict_proba(test_data[pred_cols])[:, 1]

    fpr, tpr, _ = roc_curve(p_test_num, rf_pred_prob)
    rf_auc = sk_auc(fpr, tpr)

    rf_threshold = get_spec_sens_threshold(p_test_num, rf_pred_prob)
    rf_pred_binary = (rf_pred_prob > rf_threshold).astype(int)
    rf_metrics = get_confusion_metrics(p_test_num, rf_pred_binary)

    log(
        f"Model Performance -> AUC: {rf_auc:.4f} | Sens: {rf_metrics['Sensitivity']:.4f} | "
        f"Spec: {rf_metrics['Specificity']:.4f} | Acc: {rf_metrics['Accuracy']:.4f} | "
        f"Prec: {rf_metrics['Precision']:.4f} | F1: {rf_metrics['F1_Score']:.4f}"
    )

    # --- 4f. Spatial projection ---
    log("Projecting spatial insect diversity map...")
    n_bands, gh, gw = env_stack_selected.shape
    flat = env_stack_selected.reshape(n_bands, -1).T
    flat_df = pd.DataFrame(flat, columns=selected_predictor_names)
    valid_pix = flat_df.notna().all(axis=1).to_numpy()

    rf_diversity_flat = np.full(flat_df.shape[0], np.nan)
    if valid_pix.sum() > 0:
        rf_diversity_flat[valid_pix] = rf_model.predict_proba(flat_df.loc[valid_pix])[:, 1]
    rf_diversity = rf_diversity_flat.reshape(gh, gw)

    rf_threshold_75 = np.nanquantile(rf_diversity, 0.75)
    rf_high_diversity_75 = (rf_diversity > rf_threshold_75).astype("float32")
    rf_high_diversity_75[np.isnan(rf_diversity)] = np.nan

    # Pixel area (km^2). Correct for a projected (metres) CRS - see module docstring.
    pixel_area_km2 = abs(masked_transform.a * masked_transform.e) / 1e6
    rf_area = float(np.nansum(np.where(rf_high_diversity_75 == 1, pixel_area_km2, 0)))

    # --- 4g. Reproject outputs to EPSG:4326 and save ---
    log("Reprojecting final raster outputs to EPSG:4326...")
    dst_transform, dst_w, dst_h = calculate_default_transform(ref_crs, "EPSG:4326", gw, gh, *rasterio.transform.array_bounds(gh, gw, masked_transform))

    def reproject_to_4326(arr, resampling):
        dest = np.full((dst_h, dst_w), np.nan, dtype="float32")
        reproject(
            source=arr.astype("float32"), destination=dest,
            src_transform=masked_transform, src_crs=ref_crs,
            dst_transform=dst_transform, dst_crs="EPSG:4326",
            resampling=resampling,
        )
        return dest

    rf_diversity_4326 = reproject_to_4326(rf_diversity, Resampling.bilinear)
    rf_high_diversity_75_4326 = reproject_to_4326(rf_high_diversity_75, Resampling.nearest)

    performance_summary.append({
        "Scenario": scenario_name, "AUC": rf_auc, "Sensitivity": rf_metrics["Sensitivity"],
        "Specificity": rf_metrics["Specificity"], "Accuracy": rf_metrics["Accuracy"],
        "Precision": rf_metrics["Precision"], "F1_Score": rf_metrics["F1_Score"],
        "HighDiversityArea_km2": rf_area, "N_Predictors": len(selected_predictor_names),
    })

    out_profile_4326 = {
        "driver": "GTiff", "height": dst_h, "width": dst_w, "count": 1,
        "dtype": "float32", "crs": "EPSG:4326", "transform": dst_transform, "nodata": np.nan,
    }
    with rasterio.open(os.path.join(scenario_output_dir, f"{scenario_name}_Insect_Diversity.tif"), "w", **out_profile_4326) as dst:
        dst.write(rf_diversity_4326, 1)
    with rasterio.open(os.path.join(scenario_output_dir, f"{scenario_name}_High_Diversity_Threshold_75.tif"), "w", **out_profile_4326) as dst:
        dst.write(rf_high_diversity_75_4326, 1)

    # --- 4h. Figures ---

    # Fig 1A: predictor distributions
    log("Generating variable distribution plots...")
    n_vars = len(selected_predictor_names)
    n_cols = min(3, int(np.ceil(np.sqrt(n_vars))))
    n_rows = int(np.ceil(n_vars / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 8))
    axes = np.atleast_1d(axes).flatten()
    colors = cm.get_cmap("turbo")(np.linspace(0, 1, n_vars))
    for ax, var, color in zip(axes, selected_predictor_names, colors):
        ax.hist(train_data[var].dropna(), bins=30, color=color, edgecolor="black", linewidth=0.3, alpha=0.7)
        ax.set_title(var, fontweight="bold", fontsize=9)
        apply_publication_style(ax)
    for ax in axes[n_vars:]:
        ax.axis("off")
    fig.suptitle(f"A) Selected Predictor Distributions - {scenario_name}", fontweight="bold", x=0.02, ha="left")
    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig1A_Variable_Distributions.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig1A_Variable_Distributions.png"), dpi=300)
    plt.close(fig)

    # Fig 1B: correlation heatmap
    log("Generating correlation heatmap...")
    corr_matrix = train_data[selected_predictor_names].corr()
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(corr_matrix)))
    ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(len(corr_matrix)))
    ax.set_yticklabels(corr_matrix.columns, fontsize=8)
    for r in range(len(corr_matrix)):
        for c in range(len(corr_matrix)):
            ax.text(c, r, f"{corr_matrix.iloc[r, c]:.2f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax)
    ax.set_title(f"B) Variable Correlation Matrix - {scenario_name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig1B_Correlation_Heatmap.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig1B_Correlation_Heatmap.png"), dpi=300)
    plt.close(fig)

    # Fig 2: feature importance (permutation importance ~ MeanDecreaseAccuracy)
    log("Generating feature importance plot...")
    perm = permutation_importance(rf_model, test_data[pred_cols], test_data["presence"], scoring="accuracy", random_state=RNG_SEED, n_repeats=10)
    df_imp = pd.DataFrame({"Feature": pred_cols, "Importance": perm.importances_mean}).sort_values("Importance", ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(7, 5))
    colors = cm.get_cmap("plasma")((df_imp["Importance"] - df_imp["Importance"].min()) / (df_imp["Importance"].max() - df_imp["Importance"].min() + 1e-9))
    ax.barh(df_imp["Feature"][::-1], df_imp["Importance"][::-1], color=colors[::-1], edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Importance (permutation, accuracy)")
    ax.set_ylabel("Environmental Predictor")
    apply_publication_style(ax, title=f"C) Feature Importance - {scenario_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig2_Feature_Importance.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig2_Feature_Importance.png"), dpi=300)
    plt.close(fig)

    # Fig 3: ROC curve
    log("Generating ROC curve...")
    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.plot(fpr, tpr, color="#D95F02", linewidth=1.2)
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=0.8)
    ax.scatter([1 - rf_metrics["Specificity"]], [rf_metrics["Sensitivity"]], color="red", s=60, marker="D", edgecolor="black", zorder=5)
    ax.text(0.55, 0.20, f"AUC = {rf_auc:.3f}\nThreshold = {rf_threshold:.3f}", fontsize=10, fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.8))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("1 - Specificity (False Positive Rate)")
    ax.set_ylabel("Sensitivity (True Positive Rate)")
    apply_publication_style(ax, title=f"D) ROC Curve - {scenario_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig3_ROC_Curve.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig3_ROC_Curve.png"), dpi=300)
    plt.close(fig)

    # Fig 4: response curves for top 3 predictors
    log("Generating response curves for top predictors...")
    top_3_vars = df_imp["Feature"].iloc[: min(3, len(df_imp))].tolist()
    response_frames = [partial_dependence(rf_model, train_data, pred_cols, v, grid_points=50) for v in top_3_vars]
    response_data = pd.concat(response_frames, ignore_index=True)

    n_top = len(top_3_vars)
    fig, axes = plt.subplots(1, n_top, figsize=(10, 3.5))
    axes = np.atleast_1d(axes)
    colors = cm.get_cmap("viridis")(np.linspace(0, 1, n_top))
    for ax, var, color in zip(axes, top_3_vars, colors):
        sub = response_data[response_data.Variable == var]
        ax.plot(sub.Value, sub.Probability, color=color, linewidth=1.5)
        ax.fill_between(sub.Value, 0, sub.Probability, color=color, alpha=0.2)
        ax.set_title(var, fontsize=9, fontweight="bold")
        apply_publication_style(ax)
    fig.suptitle(f"E) Response Curves (Top {n_top} Predictors) - {scenario_name}", fontweight="bold", x=0.02, ha="left")
    fig.text(0.5, 0.02, "Predictor Value", ha="center")
    axes[0].set_ylabel("Predicted Presence Probability")
    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig4_Response_Curves.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig4_Response_Curves.png"), dpi=300)
    plt.close(fig)

    # Fig 5: spatial diversity map (EPSG:4326)
    log("Generating spatial diversity map in EPSG:4326...")
    occ_sf_4326 = occ_sf_reproj.to_crs("EPSG:4326")
    fig, ax = plt.subplots(figsize=(9, 8))
    extent = rasterio.transform.array_bounds(dst_h, dst_w, dst_transform)  # (left, bottom, right, top) order varies
    left, bottom, right, top = dst_transform.c, dst_transform.f + dst_h * dst_transform.e, dst_transform.c + dst_w * dst_transform.a, dst_transform.f
    im = ax.imshow(rf_diversity_4326, extent=(left, right, bottom, top), origin="upper", cmap="viridis")
    ax.scatter(occ_sf_4326.geometry.x, occ_sf_4326.geometry.y, color="red", s=8, alpha=0.6)
    fig.colorbar(im, ax=ax, label="Diversity Index")
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    apply_publication_style(
        ax, title=f"F) Insect Diversity Map - {scenario_name}",
        subtitle="CRS: WGS 84 (EPSG:4326) | Red points = occurrence observations",
    )
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig5_Spatial_Diversity_Map.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig5_Spatial_Diversity_Map.png"), dpi=300)
    plt.close(fig)

    # Fig 6: model diagnostics (predicted vs observed + confusion matrix)
    log("Generating model diagnostic plots...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    df_pred_obs = pd.DataFrame({"Observed": p_test_num, "Predicted": rf_pred_prob})
    for cls, color in ((0, "#1B9E77"), (1, "#D95F02")):
        sub = df_pred_obs[df_pred_obs.Observed == cls]
        jitter = sub.Observed + np.random.default_rng(RNG_SEED).uniform(-0.05, 0.05, len(sub))
        ax1.scatter(jitter, sub.Predicted, color=color, alpha=0.4, s=15, label="Presence" if cls else "Absence")
        ax1.boxplot(sub.Predicted, positions=[cls], widths=0.3, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.3), showfliers=False)
    ax1.axhline(rf_threshold, color="red", linestyle="--", linewidth=0.8)
    ax1.text(0.7, rf_threshold + 0.05, f"Threshold = {rf_threshold:.3f}", fontsize=8)
    ax1.set_xlabel("Observed Class (0 = Absence, 1 = Presence)")
    ax1.set_ylabel("Predicted Probability")
    ax1.legend(title="Observed Class")
    apply_publication_style(ax1, title=f"G) Model Predictions - {scenario_name}")

    conf_counts = np.array([[rf_metrics["TN"], rf_metrics["FN"]], [rf_metrics["FP"], rf_metrics["TP"]]])
    conf_pct = 100 * conf_counts / conf_counts.sum()
    im2 = ax2.imshow(conf_counts, cmap="Greens")
    ax2.set_xticks([0, 1]); ax2.set_xticklabels(["Absence (0)", "Presence (1)"])
    ax2.set_yticks([0, 1]); ax2.set_yticklabels(["Absence (0)", "Presence (1)"])
    for r in range(2):
        for c in range(2):
            ax2.text(c, r, f"{conf_counts[r, c]}\n({conf_pct[r, c]:.1f}%)", ha="center", va="center", fontweight="bold")
    ax2.set_xlabel("Predicted"); ax2.set_ylabel("Actual")
    ax2.grid(False)
    apply_publication_style(ax2, title=f"H) Confusion Matrix - {scenario_name}")
    ax2.grid(False)

    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig6_Model_Diagnostics.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig6_Model_Diagnostics.png"), dpi=300)
    plt.close(fig)

    # Fig 7: performance metrics dashboard
    log("Generating performance metrics dashboard...")
    metrics_names = ["AUC", "Sensitivity", "Specificity", "Accuracy", "Precision", "F1-Score"]
    metrics_values = [rf_auc, rf_metrics["Sensitivity"], rf_metrics["Specificity"], rf_metrics["Accuracy"], rf_metrics["Precision"], rf_metrics["F1_Score"]]
    metrics_colors = ["#E66B4B", "#4B8DE6", "#E6D14B", "#8B4BE6", "#4BE66B", "#E64B8B"]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(metrics_names, metrics_values, color=metrics_colors, edgecolor="black", linewidth=0.5)
    for x, v in enumerate(metrics_values):
        ax.text(x, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold", fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontweight="bold")
    apply_publication_style(ax, title=f"I) Performance Metrics Summary - {scenario_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(scenario_output_dir, "Fig7_Performance_Metrics.pdf"))
    plt.savefig(os.path.join(scenario_output_dir, "Fig7_Performance_Metrics.png"), dpi=300)
    plt.close(fig)

    # --- 4i. Model summary text ---
    top_lines = "\n".join(
        f"   {k+1}. {df_imp.Feature.iloc[k]} (Importance: {df_imp.Importance.iloc[k]:.4f})"
        for k in range(min(3, len(df_imp)))
    )
    summary_text = "\n".join([
        "1. STUDY DESIGN & DATA",
        f"   Occurrence Points (Presences): {n_presence}",
        f"   Background Points (Pseudo-absences): {n_presence}",
        f"   Training Set: {len(train_data)} samples ({100*TRAIN_TEST_SPLIT:.1f}% of total)",
        f"   Test Set: {len(test_data)} samples ({100*(1-TRAIN_TEST_SPLIT):.1f}% of total)",
        f"   Initial Environmental Variables: {len(initial_vars)}",
        "",
        "2. VARIABLE SELECTION & FEATURE ENGINEERING",
        f"   VIF Threshold: {VIF_THRESHOLD:.1f}",
        f"   Final Retained Predictors ({len(selected_predictor_names)}): {', '.join(selected_predictor_names)}",
        f"   Variables Excluded by VIF ({len(excluded_vars)}): {', '.join(excluded_vars) if excluded_vars else 'None'}",
        f"   Zero-Variance Variables Removed ({len(zero_var_cols)}): {', '.join(zero_var_cols) if zero_var_cols else 'None'}",
        "",
        "3. RANDOM FOREST MODEL CONFIGURATION",
        "   Number of Trees: 500",
        f"   max_features (features per split): {max(1, int(np.floor(np.sqrt(len(pred_cols)))))}",
        f"   Random Seed: {RNG_SEED}",
        "",
        "4. MODEL PERFORMANCE EVALUATION",
        f"   Area Under ROC Curve (AUC): {rf_auc:.4f}",
        f"   Optimal Classification Threshold (Youden's J): {rf_threshold:.4f}",
        f"   Sensitivity (True Positive Rate): {rf_metrics['Sensitivity']:.4f}",
        f"   Specificity (True Negative Rate): {rf_metrics['Specificity']:.4f}",
        f"   Accuracy (Overall): {rf_metrics['Accuracy']:.4f}",
        f"   Precision (Positive Predictive Value): {rf_metrics['Precision']:.4f}",
        f"   F1-Score (Harmonic Mean): {rf_metrics['F1_Score']:.4f}",
        "   Confusion Matrix:",
        f"     True Negatives: {rf_metrics['TN']}, False Positives: {rf_metrics['FP']}",
        f"     False Negatives: {rf_metrics['FN']}, True Positives: {rf_metrics['TP']}",
        "",
        "5. SPATIAL PREDICTIONS",
        f"   High Insect Diversity Area (>75th Percentile): {rf_area:.2f} km²",
        f"   Diversity Index Range: {np.nanmin(rf_diversity_4326):.4f} to {np.nanmax(rf_diversity_4326):.4f}",
        "",
        "6. TOP 3 MOST IMPORTANT PREDICTORS",
        top_lines,
    ])
    with open(os.path.join(scenario_output_dir, "Model_Summary.txt"), "w") as f:
        f.write(summary_text)

    log(f"Completed scenario: {scenario_name}\n")

## 5. CROSS-SCENARIO COMPARISON

In [ ]:
log("\n\nGenerating Cross-Scenario Performance Comparison...")

performance_df = pd.DataFrame(performance_summary)
performance_df.to_csv(os.path.join(OUTPUT_BASE_DIR, "Scenario_Performance_Summary.csv"), index=False)

if not performance_df.empty:
    # Panel 1: main metrics comparison (grouped bar chart)
    metric_cols = ["AUC", "Sensitivity", "Specificity", "Accuracy", "Precision", "F1_Score"]
    fig = plt.figure(figsize=(14, 9))
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1])
    ax_main = fig.add_subplot(gs[0, :])

    n_scen = len(performance_df)
    n_metrics = len(metric_cols)
    bar_width = 0.8 / n_metrics
    x = np.arange(n_scen)
    set2 = plt.get_cmap("Set2")(np.linspace(0, 1, n_metrics))
    for m, (metric, color) in enumerate(zip(metric_cols, set2)):
        vals = performance_df[metric].to_numpy()
        pos = x + (m - n_metrics / 2) * bar_width + bar_width / 2
        ax_main.bar(pos, vals, width=bar_width, color=color, edgecolor="black", linewidth=0.3, label=metric)
        for xi, v in zip(pos, vals):
            ax_main.text(xi, v + 0.01, f"{v:.3f}", ha="center", fontsize=6, fontweight="bold", rotation=90)
    ax_main.set_xticks(x)
    ax_main.set_xticklabels(performance_df["Scenario"], rotation=20, ha="right", fontweight="bold")
    ax_main.set_ylim(0, 1.12)
    ax_main.set_ylabel("Metric Score")
    ax_main.legend(ncol=n_metrics, loc="upper center", bbox_to_anchor=(0.5, 1.15))
    apply_publication_style(ax_main, title="A) Cross-Scenario Model Performance Comparison")

    # Panel 2: high-diversity area
    ax_area = fig.add_subplot(gs[1, 0])
    order = performance_df.sort_values("HighDiversityArea_km2", ascending=False)
    ax_area.bar(order["Scenario"], order["HighDiversityArea_km2"], color=PALETTE_SCENARIO[: len(order)], edgecolor="black", linewidth=0.5)
    for xi, v in enumerate(order["HighDiversityArea_km2"]):
        ax_area.text(xi, v, f"{v:.0f} km²", ha="center", va="bottom", fontweight="bold", fontsize=8)
    plt.setp(ax_area.get_xticklabels(), rotation=20, ha="right", fontweight="bold")
    apply_publication_style(ax_area, title="B) High Diversity Area Estimates")

    # Panel 3: number of predictors
    ax_pred = fig.add_subplot(gs[1, 1])
    order2 = performance_df.sort_values("N_Predictors", ascending=False)
    ax_pred.bar(order2["Scenario"], order2["N_Predictors"], color=PALETTE_SCENARIO[: len(order2)], edgecolor="black", linewidth=0.5)
    for xi, v in enumerate(order2["N_Predictors"]):
        ax_pred.text(xi, v, f"{v:d}", ha="center", va="bottom", fontweight="bold", fontsize=8)
    plt.setp(ax_pred.get_xticklabels(), rotation=20, ha="right", fontweight="bold")
    apply_publication_style(ax_pred, title="C) Model Complexity (Retained Predictors)")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE_DIR, "Fig_Cross_Scenario_Comparison.pdf"))
    plt.savefig(os.path.join(OUTPUT_BASE_DIR, "Fig_Cross_Scenario_Comparison.png"), dpi=300)
    plt.close(fig)

    log("Generating final summary tables...")
    summary_table = performance_df.copy()
    for col in ["AUC", "Sensitivity", "Specificity", "Accuracy", "Precision", "F1_Score"]:
        summary_table[col] = summary_table[col].map(lambda v: f"{v:.4f}")
    summary_table["HighDiversityArea_km2"] = summary_table["HighDiversityArea_km2"].map(lambda v: f"{v:.2f}")
    summary_table.to_csv(os.path.join(OUTPUT_BASE_DIR, "Supplementary_Table_Model_Metrics.csv"), index=False)

with open(os.path.join(OUTPUT_BASE_DIR, "SessionInfo.txt"), "w") as f:
    f.write(f"Analysis Date: {datetime.now():%Y-%m-%d %H:%M:%S}\n\n")
    f.write("Python Session Information:\n")
    f.write(f"Python: {sys.version}\n")
    f.write(f"Platform: {platform.platform()}\n")
    for mod in (np, pd):
        f.write(f"{mod.__name__}: {mod.__version__}\n")

log("")